# **FASE #2:** **Motor de Procesamiento Digital de Señales (DSP) y DataLoaders**

### **BLOQUE: Parseo y Unificación**

Este índice (en formato Pandas DataFrame) actuará como el "mapa de rutas" que el `DataLoader` de PyTorch consultará iterativamente para cargar, aplicar transformaciones en vuelo y enviar los tensores a la GPU de manera eficiente.


In [1]:
import pandas as pd
from pathlib import Path
from typing import List, Union

# Mapeo taxonómico constante del sistema USAR V2.0
LABEL_MAP = {
    "Noise_General": 0,
    "Target_Animal": 1,
    "Target_Rhythmic": 2,
    "Target_Voice": 3
}

def parsear_y_unificar_datasets(lista_directorios_raiz: List[Union[str, Path]]) -> pd.DataFrame:
    """
    Escanea recursivamente una lista de directorios raiz para indexar tensores de audio.

    La funcion busca archivos con extension .wav y extrae sus metadatos basandose en
    la jerarquia de carpetas. Filtra estrictamente los hallazgos asegurando que 
    pertenezcan a la taxonomia definida en LABEL_MAP. Los archivos en directorios 
    no reconocidos son reportados y descartados.

    Parameters
    ----------
    lista_directorios_raiz : List[Union[str, Path]]
        Lista de rutas (strings o objetos Path) correspondientes a las carpetas 
        raiz de los datasets (ej. ['dataset_preprocesado', 'Dataset_Feedback']).

    Returns
    -------
    pd.DataFrame
        DataFrame estructurado con las columnas: 'ruta_absoluta', 'dataset_origen', 
        'categoria' y 'label_idx'. Retorna un DataFrame vacio si no hay coincidencias.
    """
    datos_indexados = []

    for directorio in lista_directorios_raiz:
        ruta_raiz = Path(directorio)
        
        # Validacion de existencia del directorio
        if not ruta_raiz.exists() or not ruta_raiz.is_dir():
            print(f"[ADVERTENCIA] El directorio raiz especificado no existe o no es valido: {ruta_raiz}")
            continue
            
        # Busqueda recursiva de archivos .wav
        for archivo_wav in ruta_raiz.rglob('*.wav'):
            if not archivo_wav.is_file():
                continue
                
            try:
                # Extraccion de metadatos desde la estructura del arbol de directorios
                categoria = archivo_wav.parent.name
                dataset_origen = archivo_wav.parent.parent.name
                
                # Validacion estricta contra la taxonomia aprobada
                if categoria not in LABEL_MAP:
                    print(f"[ADVERTENCIA] Categoria anomala '{categoria}' detectada en {archivo_wav.name}. Archivo ignorado.")
                    continue
                    
                # Resolucion de ruta absoluta para evitar punteros rotos en el DataLoader
                ruta_absoluta = str(archivo_wav.resolve())
                label_idx = LABEL_MAP[categoria]
                
                datos_indexados.append({
                    "ruta_absoluta": ruta_absoluta,
                    "dataset_origen": dataset_origen,
                    "categoria": categoria,
                    "label_idx": label_idx
                })
                
            except Exception as e:
                # Captura de errores de lectura de pathing o permisos
                print(f"[ADVERTENCIA] Error procesando el archivo {archivo_wav}: {str(e)}")
                continue

    # Construccion y retorno del DataFrame analitico
    df_unificado = pd.DataFrame(datos_indexados, columns=[
        "ruta_absoluta", 
        "dataset_origen", 
        "categoria", 
        "label_idx"
    ])
    
    return df_unificado
